# Module import

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import xarray as xr
import polars as pl
import os
import importlib
import re

from toolbox.steps.base_step import BaseStep, register_step
from toolbox.steps.custom import find_profiles_pitch as fpP

[Discovery] Scanning for step modules in /Users/hh2n18/Documents/GitHub/pelagos-pyHans/src/toolbox/steps/custom
[Discovery] Importing step module: toolbox.steps.custom.find_profiles_pitch
[Discovery] Importing step module: toolbox.steps.custom.blank_step
[Discovery] Importing step module: toolbox.steps.custom.apply_qc
[Discovery] Importing step module: toolbox.steps.custom.derive_ctd
[Discovery] Importing step module: toolbox.steps.custom.find_profiles
[Discovery] Importing step module: toolbox.steps.custom.export
[Discovery] Importing step module: toolbox.steps.custom.gen_data
[Discovery] Importing step module: toolbox.steps.custom.profile_direction
[Discovery] Importing step module: toolbox.steps.custom.interpolate_data
[Discovery] Importing step module: toolbox.steps.custom.load_data
[Discovery] Importing step module: toolbox.steps.custom.write_report
[Discovery] Importing step module: toolbox.steps.custom.variables.salinity
[Discovery] Importing step module: toolbox.steps.custom.va

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


[Discovery] Importing step module: toolbox.steps.custom.qc.impossible_date_qc
[Discovery] Registered step: Find Profiles
[Discovery] Registered step: Blank Step
[Discovery] Registered step: Apply QC
[Discovery] Registered step: Derive CTD
[Discovery] Registered step: Data Export
[Discovery] Registered step: Generate Data
[Discovery] Registered step: Find Profile Direction
[Discovery] Registered step: Interpolate Data
[Discovery] Registered step: Load OG1
[Discovery] Registered step: Write Data Report
[Discovery] Registered step: Salinity Adjustment
[Discovery] Registered step: BBP from Beta
[Discovery] Registered step: Isolate BBP Spikes
[Discovery] Registered step: Chla Deep Correction
[Discovery] Registered step: Chla Quenching Correction
[Discovery] Registered step: Derive Uncalibrated Phase
[Discovery] Registered step: Derive Optode Temperature
[Discovery] Registered step: Phase Pressure Correction
[Discovery] Registered step: Derive Calibrated Phase
[Discovery] Registered step: De

# Data import

In [3]:
file = "/Users/hh2n18/Documents/GitHub/pelagos-pyHans/test_data/Cabot_645_Processed.nc"

df = xr.open_dataset(file)

df = pl.from_pandas(
    df[["TIME", "PRES", "GLIDER_PITCH"]].to_dataframe().reset_index(),
    nan_to_null=False,
)

# Profile finding

In [ ]:
importlib.reload(fpP)

gradient_thresholds=[0.005, -0.025]
custom_thresholds = [0.0, -0.025]
custom_column = "GLIDER_PITCH"

profiles_df = fpP.find_profiles(
    df,
    gradient_thresholds=gradient_thresholds,
    filter_win_sizes=["20s", "10s"],
    time_col="TIME",
    depth_col="PRES",
    transect_duration="10m",
    transect_depth_range=[10.0],
    transect_depth_bottom_limits=None,
    cust_col=custom_column,
    cust_gradient_thresholds=custom_thresholds,
    surface_depth = 3
)

# Plotting

In [23]:
# Split profiles vs not profiles
profiles = profiles_df.drop_nans().filter(pl.col("is_profile").cast(pl.Boolean))
not_profiles = profiles_df.drop_nans().filter(pl.col("is_profile").cast(pl.Boolean).not_())
transects = profiles_df.drop_nans().filter(pl.col("is_transect").cast(pl.Boolean))
not_transects = profiles_df.drop_nans().filter(pl.col("is_transect").cast(pl.Boolean).not_())
 
# === Plotting ===

mpl.use('tkagg')
fig, axs = plt.subplots(5, 1, figsize=(18, 10), height_ratios=[3, 3, 1, 3, 3], sharex=True)
 
axs[0].set(xlabel="Time", ylabel="Interpolated Depth")
axs[1].set(xlabel="Time", ylabel="Vertical Velocity")
axs[2].set(xlabel="Time", ylabel="Profile Number")
axs[3].set(xlabel="Time", ylabel=custom_column)   # <<< custom col
axs[4].set(xlabel="Time", ylabel=custom_column)   # <<< custom col
 
# Profile vs not profile (depth + velocity)
for data, col, label in zip([profiles, not_profiles, transects],
                            ["tab:blue", "tab:red", "tab:orange"],
                            ["Profile", "Not Profile", "Transect"]):
    axs[0].plot(data["TIME"].to_numpy(), -data["INTERP_PRES"].to_numpy(),
                marker=".", markersize=1, ls="", c=col, label=label)
    axs[1].plot(data["TIME"].to_numpy(), data["smooth_grad"].to_numpy(),
                marker=".", markersize=1, ls="", c=col, label=label)
 
# Raw gradients + thresholds
axs[1].plot(profiles_df["TIME"].to_numpy(), profiles_df["grad"].to_numpy(),
            c="k", alpha=0.1, label="Raw Velocity")
for val, label in zip(gradient_thresholds, ["Gradient Thresholds", None]):
    axs[1].axhline(val, ls="--", color="gray", label=label)
 
# Profile numbers
axs[2].plot(profiles_df["TIME"].to_numpy(), profiles_df["profile_num"].to_numpy(), c="gray")
#axs[2].plot(profiles_df["TIME"].to_numpy(), profiles_df["profile_num_new"].to_numpy(), c="red")
 
# Custom column (pitch)
if custom_column in profiles_df.columns:
    axs[3].plot(profiles_df["TIME"].to_numpy(),
                profiles_df[custom_column].to_numpy(),
                c="purple", marker=".", markersize=1, ls="", label=custom_column)
    axs[3].legend(loc="right")
 
# Custom column (pitch)
if custom_column in profiles_df.columns:
    axs[4].plot(profiles_df["TIME"].to_numpy(),
                -profiles_df[f'smooth_INTERP_{custom_column}'].to_numpy()*profiles_df['smooth_grad'].to_numpy(),
                c="purple", marker=".", markersize=1, ls="", label=custom_column)
    axs[4].legend(loc="right")
    axs[4].set_ylim([0, 0.05])
for val, label in zip(custom_thresholds, ["Gradient Thresholds", None]):
    axs[4].axhline(val, ls="--", color="gray", label=label)
 
for ax in axs[:2]:
    ax.legend(loc="upper right")
 
plt.tight_layout()
plt.show(block=True)

In [15]:
mpl.use('tkagg')
# ── Phase science diagnostic figure ──────────────────────────────────────
_PHASE_NAMES = {
    0: "unknown", 1: "ascent",     2: "descent",   3: "surfacing",
    4: "parking", 5: "inflection", 6: "propelled",  7: "transition",
}
_PHASE_COLORS = {
    0: "lightgray", 1: "tab:blue", 2: "tab:red",   3: "tab:cyan",
    4: "tab:olive", 5: "gold",     6: "tab:green",  7: "tab:purple",
} 
_BASE_S = 5  # base scatter marker area (points²); scales proportionally with zoom

_phase_df = profiles_df.drop_nulls()
_time  = _phase_df["TIME"].to_numpy()
_depth = -_phase_df["INTERP_PRES"].to_numpy()
_phase = _phase_df["PHASE_SCI"].to_numpy().astype(int)

_plot_grad = True
_has_cust = custom_column in profiles_df.columns
n_rows = 4 if _has_cust and _plot_grad else 3 if _has_cust else 2
height_ratios = [4, 1, 2, 2] if _has_cust and _plot_grad else [4, 1, 2] if _has_cust else [4, 1]
fig_phase, axs_phase = plt.subplots(
    n_rows, 1, figsize=(18, 7), height_ratios=height_ratios, sharex=True,
)
axs_phase[0].set(ylabel="Depth")
axs_phase[1].set(xlabel="Time", ylabel="Phase")
axs_phase[1].set_yticks(list(_PHASE_NAMES.keys()))
axs_phase[1].set_yticklabels([f"{k}  {v}" for k, v in _PHASE_NAMES.items()])
axs_phase[1].set_ylim(-0.5, 7.5)

for _pid, _pname in _PHASE_NAMES.items():
    _mask = _phase == _pid
    if not _mask.any():
        continue
    _c = _PHASE_COLORS[_pid]
    axs_phase[0].scatter(
        _time[_mask], _depth[_mask], s=_BASE_S, c=_c,
        label=f"{_pid}  {_pname}",
    )
    axs_phase[1].scatter(
        _time[_mask], np.full(_mask.sum(), _pid), s=_BASE_S, c=_c,
    )

if _has_cust:
    axs_phase[2].plot(
        profiles_df["TIME"].to_numpy(),
        -profiles_df[f"smooth_INTERP_{custom_column}"].to_numpy() * profiles_df["smooth_grad"].to_numpy(),
        c="purple", marker=".", markersize=1, ls="", label=custom_column,
    )
    for val, label in zip(custom_thresholds, ["Gradient Thresholds", None]):
        axs_phase[2].axhline(val, ls="--", color="gray", label=label)
    axs_phase[2].legend(loc="right")
    axs_phase[2].set_ylim([0, 0.05])

if _plot_grad:
# Raw gradients + thresholds
    axs_phase[3].plot(profiles_df["TIME"].to_numpy(), profiles_df["smooth_grad"].to_numpy(),
                c="k", alpha=0.1, label="Raw Velocity")
    for val, label in zip(gradient_thresholds, ["Gradient Thresholds", None]):
        axs_phase[3].axhline(val, ls="--", color="gray", label=label)
    axs_phase[3].legend(loc="right")
    axs_phase[3].set_ylim([-0.5, 0.5])

axs_phase[0].legend(loc="upper right", markerscale=2)
fig_phase.tight_layout()


def _rescale_phase_markers(ax, all_axes, x0_range, base_s=_BASE_S):
    """Rescale scatter markers proportionally to the current x-axis zoom level."""
    span = ax.get_xlim()[1] - ax.get_xlim()[0]
    if span <= 0:
        return
    new_s = float(np.clip(base_s * (x0_range / span), base_s * 0.25, base_s * 100))
    for a in all_axes:
        for coll in a.collections:
            coll.set_sizes([new_s])
    ax.figure.canvas.draw_idle()


_x0_range = axs_phase[0].get_xlim()[1] - axs_phase[0].get_xlim()[0]
axs_phase[0].callbacks.connect(
    "xlim_changed",
    lambda ax: _rescale_phase_markers(ax, axs_phase, _x0_range),
)
plt.show(block=True)